# Cost landscapes
**Quantum Computing · Week 7 (Variational quantum circuits) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w7d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Give a circuit knobs and a **cost** to minimize, and every knob setting gets a value: a **cost landscape**.
Today's cost is an energy, $E(\boldsymbol\theta) = \langle\psi(\boldsymbol\theta)|H|\psi(\boldsymbol\theta)\rangle$
with $H = Z_0Z_1 + 0.5\,X_0 + 0.5\,X_1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

H = SparsePauliOp.from_list([("ZZ", 1.0), ("IX", 0.5), ("XI", 0.5)])   # Qiskit reads Pauli strings right to left
print("energies of H (eigenvalues):", np.linalg.eigvalsh(H.to_matrix()).round(4))

t1, t2 = Parameter("θ1"), Parameter("θ2")
circuit = QuantumCircuit(2)
circuit.ry(t1, 0)
circuit.ry(t2, 1)
circuit.cx(0, 1)

estimator = StatevectorEstimator()
grid = np.linspace(-np.pi, np.pi, 61)
points = np.array([[a, b] for a in grid for b in grid])
E = estimator.run([(circuit, H, points)]).result()[0].data.evs.reshape(len(grid), len(grid))
print(f"lowest energy on the grid: {E.min():.3f}   ground state: {-np.sqrt(2):.3f}")

plt.figure(figsize=(5, 4.2))
plt.contourf(grid, grid, E.T, levels=30, cmap="Purples_r")
plt.colorbar(label="energy")
plt.xlabel(r"$\theta_1$")
plt.ylabel(r"$\theta_2$")
plt.title("The cost landscape")
plt.show()

## Every slice is a sine wave
Fix all knobs but one. Because that knob sits in a single rotation gate, the cost is exactly
$C(\theta) = a\cos\theta + b\sin\theta + c$. Three evaluations pin down $a$, $b$ and $c$, and then the
slice's minimum is known exactly.

In [ ]:
def slice_energy(theta, fixed=2.5):
    return estimator.run([(circuit, H, [theta, fixed])]).result()[0].data.evs.item()

c0, cp, cm = slice_energy(0.0), slice_energy(np.pi / 2), slice_energy(-np.pi / 2)
c = (cp + cm) / 2
b = (cp - cm) / 2
a = c0 - c
xs = np.linspace(-np.pi, np.pi, 200)
exact = [slice_energy(x) for x in xs[::8]]
plt.plot(xs, a * np.cos(xs) + b * np.sin(xs) + c, color="#ffb08a", lw=3, label="sine from 3 points")
plt.plot(xs[::8], exact, "o", color="#7b5cfa", ms=5, label="exact values")
plt.plot([0, np.pi / 2, -np.pi / 2], [c0, cp, cm], "s", color="#2c2842", ms=8, label="the 3 evaluations")
plt.xlabel(r"$\theta_1$  ($\theta_2$ fixed at 2.5)")
plt.ylabel("energy")
plt.legend()
plt.show()
print(f"slice minimum at theta1 = {np.arctan2(-b, -a):.3f}, energy {c - np.hypot(a, b):.3f}")

## Minima, good and bad
Two ways down. **Small steps** follow the local slope (gradient descent; the slope of each sine slice comes
from two evaluations, more on that tomorrow). **Jumps** go straight to the bottom of a slice, knob by knob.

In [ ]:
def energy(theta):
    return estimator.run([(circuit, H, theta)]).result()[0].data.evs.item()

def small_steps(start, lr=0.2, steps=150):
    theta = np.array(start, dtype=float)
    for _ in range(steps):
        slope = [(energy(theta + s) - energy(theta - s)) / 2 for s in (np.array([np.pi / 2, 0]), np.array([0, np.pi / 2]))]
        theta -= lr * np.array(slope)
    return energy(theta)

def jumps(start, rounds=6):
    theta = np.array(start, dtype=float)
    for _ in range(rounds):
        for k in range(2):
            def f(x):
                trial = theta.copy()
                trial[k] = x
                return energy(trial)
            c0, cp, cm = f(0.0), f(np.pi / 2), f(-np.pi / 2)
            c = (cp + cm) / 2
            theta[k] = np.arctan2(-(cp - cm) / 2, -(c0 - c))
    return energy(theta)

for start in [(0.3, 0.3), (-2.0, 2.0), (-1.5, -2.5)]:
    print(f"start {start}: small steps end at {small_steps(start):.3f}, jumps end at {jumps(start):.3f}")

From some starts, small steps settle at −1.000: a **local minimum**. Every small step from there goes
uphill, yet it isn't the bottom. The jumps, which look at a whole slice at once, leap out of it here.
Bigger landscapes have many more such traps.

## Your turn: the landscape through shot noise
On hardware every energy is estimated from shots. Recompute the slice along $\theta_1$ (with $\theta_2 = 2.5$)
from 100 and from 10,000 shots, using `StatevectorEstimator(default_precision=1/np.sqrt(shots))`, and plot
both against the exact curve. How much do the 100-shot values wobble?

In [ ]:
xs_coarse = np.linspace(-np.pi, np.pi, 41)
exact_slice = estimator.run([(circuit, H, [[x, 2.5] for x in xs_coarse])]).result()[0].data.evs
for shots in [100, 10000]:
    noisy_estimator = estimator     # TODO: StatevectorEstimator(default_precision=1 / np.sqrt(shots), seed=1)
    values = exact_slice            # TODO: noisy_estimator.run(...) for the same points
    plt.plot(xs_coarse, values, "o", ms=4, label=f"{shots} shots")
plt.plot(xs_coarse, exact_slice, color="#2c2842", label="exact")
plt.legend()
plt.show()

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
xs_coarse = np.linspace(-np.pi, np.pi, 41)
exact_slice = estimator.run([(circuit, H, [[x, 2.5] for x in xs_coarse])]).result()[0].data.evs
for shots, color in [(100, "#ff8a50"), (10000, "#7b5cfa")]:
    noisy_estimator = StatevectorEstimator(default_precision=1 / np.sqrt(shots), seed=1)
    values = noisy_estimator.run([(circuit, H, [[x, 2.5] for x in xs_coarse])]).result()[0].data.evs
    plt.plot(xs_coarse, values, "o", ms=4, color=color, label=f"{shots} shots")
    print(f"{shots:6d} shots: typical error {np.std(values - exact_slice):.3f}")
plt.plot(xs_coarse, exact_slice, color="#2c2842", label="exact")
plt.xlabel(r"$\theta_1$")
plt.ylabel("energy")
plt.legend()
plt.show()
print("100 shots blur the landscape by about 0.1: comparable to the depth of shallow valleys.")